# Intergration: All zikv & mock 10X data to merge data

In [ ]:
ls -l /mnt/d/WY_single_cell_analysis/data/raw

total 0
drwxrwxrwx 1 bio02 bio02 4096 Jul 28 11:07 MoCK1/
drwxrwxrwx 1 bio02 bio02 4096 Jul 28 11:07 MoCK2/
drwxrwxrwx 1 bio02 bio02 4096 Jul 28 11:08 MoCK3/
drwxrwxrwx 1 bio02 bio02 4096 Jul 28 11:08 Zikv1/
drwxrwxrwx 1 bio02 bio02 4096 Jul 28 11:08 Zikv2/
drwxrwxrwx 1 bio02 bio02 4096 Jul 28 11:08 Zikv3/


In [ ]:
import scanpy as sc
import scvi
import pandas as pd
import numpy as np
from pathlib import Path

# store QC summary per sample
stats = []

def pp_10x(mtx_dir: Path):
    """
    Perform QC, Doublet removal (SOLO), and save filtered AnnData per sample.
    """
    sample_name = mtx_dir.parent.name
    print(f"\n--- Processing Sample: {sample_name} ---")
    
    # 1. Load raw data 
    adata_raw = sc.read_10x_mtx(mtx_dir, var_names="gene_symbols", cache=True)
    adata_raw.var_names_make_unique()
    raw_count = adata_raw.n_obs
    print(f"🔍 Raw cell count: {raw_count}")

    # 2. SOLO data for Doublet Detection 
    adata_for_solo = adata_raw.copy()
    sc.pp.filter_genes(adata_for_solo, min_cells=3)
    # seurat_v3 flavor is ideal for raw counts (no log-norm required)
    sc.pp.highly_variable_genes(adata_for_solo, n_top_genes=2000, subset=True, flavor='seurat_v3')
    
    # Train scVI model
    scvi.model.SCVI.setup_anndata(adata_for_solo)
    vae = scvi.model.SCVI(adata_for_solo)
    vae.train(check_val_every_n_epoch=1)
    solo = scvi.external.SOLO.from_scvi_model(vae) # Run SOLO
    solo.train()

    # Define model saving directory
    model_dir = Path(f"/mnt/d/WY_single_cell_analysis/data/proceed/models/{sample_name}")
    model_dir.mkdir(parents=True, exist_ok=True)
    # Save the scVI model
    vae.save(model_dir / "scvi_model", overwrite=True)
    # Save the SOLO model
    solo.save(model_dir / "solo_model", overwrite=True)
    print(f"💾 Models for {sample_name} saved to: {model_dir}")
    
    # Predict doublets
    df = solo.predict()
    df['prediction'] = solo.predict(soft=False)
    # Standardize index (removing '-1' suffix if present)
    df['dif'] = df.doublet - df.singlet
    df.index = df.index.map(lambda x: x[:-2] if x.endswith("-1") else x)
    doublets = df[(df.prediction == 'doublet') & (df.dif > 1)]
    
    # 3. Filter Doublets from the full dataset
    adata = adata_raw[~adata_raw.obs_names.isin(doublets.index)].copy()
    adata.var_names_make_unique()
    after_doublet = adata.n_obs
    print(f"🧹 Cells remaining after doublet removal: {after_doublet}")

    # 4. Quality Control (QC)
    # Define Mitochondrial and ZIKV virus genes
    adata.var['mt'] = adata.var_names.str.upper().str.startswith("MT-")
    # Identify ZIKV genes (Ensure 'ZIKV' matches your reference genome prefix)
    adata.var['zikv'] = adata.var_names.str.contains("bingdu", case=False)
    
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'zikv'], percent_top=None, log1p=False, inplace=True)

    # Define filtering thresholds (98th percentile for genes, 20% for MT)
    upper_lim = np.quantile(adata.obs.n_genes_by_counts.values, 0.98)
    adata = adata[(adata.obs.n_genes_by_counts < upper_lim) & 
                  (adata.obs.n_genes_by_counts > 200) &
                  (adata.obs.pct_counts_mt < 20)].copy()
    
    after_qc = adata.n_obs
    print(f"✅ After QC: {after_qc} cells ({after_qc/adata_raw.n_obs:.2%})\n")
    
    # 5. Add Metadata
    adata.obs['sample'] = sample_name
    adata.obs['group'] = 'ZIKV' if 'Zikv' in sample_name.lower() else 'MOCK'
    
    # 6. Save filtered adata (Checkpoint) ---
    save_path = Path(f"/mnt/d/WY_single_cell_analysis/data/proceed/qc_filtered/qc_filtered_neo_{sample_name}.h5ad")
    
    # Create the directory if it doesn't exist
    save_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Write the file (only once)
    adata.write(save_path)
    print(f"💾 Sample saved to: {save_path}")

    # --- 7. Record Statistics ---
    stats.append({
        "Sample": sample_name,
        "Group": adata.obs['group'][0],
        "Raw_Cells": raw_count,
        "Post_Doublet": after_doublet,
        "Post_QC": after_qc,
        "Retention_Rate": f"{(after_qc / raw_count):.2%}"
    })

    return adata

In [ ]:
# ---
base_dir = Path("/mnt/d/WY_single_cell_analysis/data/raw")
all_adatas = []

for sample_dir in sorted(base_dir.iterdir()):
    mtx_dir = sample_dir / "filtered_feature_bc_matrix"
    if mtx_dir.exists():
        adata = pp_10x(mtx_dir)
        all_adatas.append(adata)




--- Processing Sample: MoCK1 ---
🔍 Raw cell count: 12675


/home/bio02/miniconda3/envs/sc2/lib/python3.11/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/bio02/miniconda3/envs/sc2/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/bio02/miniconda3/envs/sc2/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the val

Training:   0%|          | 0/400 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=400` reached.


INFO     Creating doublets, preparing SOLO model.                                                                  


/home/bio02/miniconda3/envs/sc2/lib/python3.11/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/bio02/miniconda3/envs/sc2/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/bio02/miniconda3/envs/sc2/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the val

... (56 outputs left out)

Monitored metric validation_loss did not improve in the last 30 records. Best score: 0.509. Signaling Trainer to stop.
💾 Models for Zikv3 saved to: /mnt/d/WY_single_cell_analysis/data/proceed/models/Zikv3


/home/bio02/miniconda3/envs/sc2/lib/python3.11/site-packages/torch/utils/_contextlib.py:116: UserWarning: Prior to scvi-tools 1.1.3, `SOLO.predict` with `soft=True` (the default option) returned logits instead of probabilities. This behavior has since been corrected to return probabiltiies. The previous behavior can be replicated by passing in `return_logits=True`.
  return func(*args, **kwargs)
/home/bio02/miniconda3/envs/sc2/lib/python3.11/site-packages/torch/utils/_contextlib.py:116: UserWarning: Prior to scvi-tools 1.1.3, `SOLO.predict` with `soft=True` (the default option) returned logits instead of probabilities. This behavior has since been corrected to return probabiltiies. The previous behavior can be replicated by passing in `return_logits=True`.
  return func(*args, **kwargs)


🧹 Cells remaining after doublet removal: 5007
✅ After QC: 4717 cells (94.21%)

💾 Sample saved to: /mnt/d/WY_single_cell_analysis/data/proceed/qc_filtered/qc_filtered_neo_Zikv3.h5ad


/tmp/ipykernel_855/1340534921.py:93: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "Group": adata.obs['group'][0],


In [ ]:
# restar Kernel to continue
import scanpy as sc
import scvi
import pandas as pd
from pathlib import Path

data_dir = Path("/mnt/d/WY_single_cell_analysis/data/proceed/qc_filtered")
h5ad_files = sorted(list(data_dir.glob("qc_filtered_neo_*.h5ad")))
adatas = [sc.read_h5ad(f) for f in h5ad_files]

# adata_combined
adata_combined = sc.concat(adatas, join="inner", label="batch")
adata_combined.obs_names_make_unique()

print(f"Combined cells: {adata_combined.n_obs}")

# 接下来继续运行我之前提供给你的 Step 2 - Step 8 的集成代码...

Combined cells: 61344


/home/bio02/miniconda3/envs/sc2/lib/python3.11/site-packages/anndata/_core/anndata.py:1774: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
# 重复的 Barcode 加上 -0, -1
adata_combined.obs_names_make_unique()

# 验证一下是否还有重复 (应该返回 False)
print(f"是否有重复的观测名: {adata_combined.obs_names.duplicated().any()}")

是否有重复的观测名: False


In [ ]:
# 检查关键的 obs 列名
print("当前元数据列:", adata_combined.obs.columns.tolist())

# 检查病毒基因计数之和，确保 ZIKV 样本的数据进来了
if 'total_counts_zikv' in adata_combined.obs:
    print(f"总病毒基因计数: {adata_combined.obs['total_counts_zikv'].sum()}")
else:
    print("⚠️ 警告: 未找到 total_counts_zikv 列，请检查之前的合并逻辑！")

当前元数据列: ['n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_zikv', 'pct_counts_zikv', 'sample', 'group', 'batch']
总病毒基因计数: 823031.0
